# Overview   

### Pipeline  
1. Compile 10 raw per-feature CSVs into a usable tensor  
    - Read the last 288 columns from each csv and check for NaN or inf values  
    - Convert labels into a binary classification of SEP and NSEP  
    - Stack in resulting X tensor of shape (N, 288, 10) and a y tensor of shape (N,) 

2. Split dataset chronologically (70/10/20) while maintaining class imbalance across sets  

3. Normalize dataset per feature (Two separate approaches in codebase until I settle on one over the other)  
    * Hybrid normalization where the applied normalization is based on the statistical behavior of the channel  
        * $F$ : Log + Z-score  
        * $N_p$ : Log + Z-score  
        * $P_4$ : Log + Z-score
        * $P_5$ : Log + Z-score
        * $P_6$ : Log + Z-score
        * $T_p$ : Log + Z-score
        * $V$ : Z-score
        * $V_x$ : Z-score
        * $Xl$ : Log + Z-score
        * $Xs$ : Log + Z-score

4. Test baseline classifiers (using hybrid normalization)  
    * Logistic Regression 
        * Tune the C, class_weight, and threshold parameters on validation set, then test on test set
        * Tune the classification threshold based on the ROC curve to: 0.4818
        * TSS: 0.42946
    * XGBoost  
        * Tune the learn_rate, scale_pos_weight, max_depth, min_child_weight, and colsample_bytree, then test on test set
        * Tune the classification threshold based on the ROC curve to: 0.00720
        * TSS: 0.64839
    * GRU
        * Tune the units, dropout, layers, learning rate, and batch size, then test on test set
        * Tune the classification threshold based on the ROC curve to: 0.6647
        * TSS: 0.60801 
    * SVM
        * Extract catch22 features and tune the class weight, gamma, and kernel, then test on test set
        * TSS: 0.48763

5. Triplet Loss 
    * Configuration
        * 1D ConvNet Encoder with Leaky ReLU, max pooling, and L2 normalization
        * Triplet loss with Euclidean distance
        * Adam optimizer
        * Logistic Regression classifier
        * NSEP anchors are chosen from the 1000 NSEP examples closest to any SEP window
    * Current Issues
        * Overfitting   
    * Tuning hyperparameters 
        * weight_decay: 0.012, hidden_ch: 32, emb_dim: 32, depth: 5, sep_fraction: 0.5, margin: 0.1
        * val TSS after 1000 learning steps is 0.670 ± 0.06


* Always train/val on multiple seeds and take mean performance to determine best hyperparameter
* Tuning to these parameters: weight_decay: 0.012, hidden_ch: 32, emb_dim: 32, depth: 5, sep_fraction: 0.5, margin: 0.1 > The val results look good but test TSS is low (around 0.30).
* Tuning LR threshold based on roc curve. Results didn't increase that much though
* TSS often jumps during training steps (0.63 at step 300 and the 0.58 on step 400 and the back up to 0.62 at step 500). Trying picking hyperparameters based on ROC-AUC since it seems more stable during learning steps, but still using TSS as main metric to see how well it performs overall.
* Recalculate table of anchors from current embeddings every N steps -> TSS went way down.
* Try selecting NSEP positives from among N closest NSEPs
* Dropout
* Implement basic classifiers with little success


To Try:
* different anchor mining strategy 
* catch 22 feature extraction
* RUS
* K cross-fold training set
* train margin like i do thr


# Setup

In [264]:
import os
import copy
import warnings
import random
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from itertools import product
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, average_precision_score, brier_score_loss
from sklearn.utils import shuffle
from sklearn.svm import SVC, LinearSVC
from sklearn.exceptions import ConvergenceWarning
from sklearn.neighbors import KNeighborsClassifier
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
from joblib import Parallel, delayed
from scipy.special import expit
from torch.utils.data import TensorDataset, DataLoader, Sampler, Dataset, WeightedRandomSampler

In [265]:
# Random state seed
RAND_STATE = 42

# Compile Data

In [266]:
DATA_DIR = Path("../dataset")
csv_files = sorted(DATA_DIR.glob('*.gz'))

channel_names = []
data_list = []
y_tensor_raw = None
onset = None
    
for i, file in enumerate(csv_files):
    # Track channel order
    channel_names.append(file.name.split('.')[0])

    df = pd.read_csv(file)

    # Confirm time col order is same as rest of file and is chronological (col 2)
    file_onset = pd.to_datetime(df.iloc[:, 2])
    if file_onset.isna().any():
        raise ValueError(f"{file.name}: {file_onset.isna().sum()} missing/invalid onset times")

    if i == 0:
        # Check the first file that it is in chronological order
        if not file_onset.is_monotonic_increasing:
            raise ValueError(f"{file.name}: rows are NOT in chronological order")
        onset = file_onset
    else:
        # Compare that it matches the first file
        if not np.array_equal(file_onset.values, onset.values):
            raise ValueError(f"{file.name}: event order differs from {csv_files[0].name}")
        
    # Extract 288 features from end and convert to numpy
    only_features = df.iloc[:,-288:]
    features_np = only_features.to_numpy().astype(np.float64)

    # Check for NaN or inf values
    if np.isnan(features_np).any():
        raise ValueError(f"{file.name}: contains NaN values")
    if np.isinf(features_np).any():
       raise ValueError(f"{file.name}: contains inf values")

    data_list.append(features_np)

    # labels will be same for every file, so just extract from the first one
    if i == 0:
        labels = df.iloc[:,0]

        # Check for missing labels
        if labels.isna().any():
            raise ValueError(f"{file.name}: {labels.isna().sum()} missing labels")

        # If 'NSEP', label is 0
        # Otherwise, label is 1 (gt10, gt30, gt60, gt100)
        y_tensor_raw = np.where(labels == 'NSEP', 0, 1)

X_tensor_raw = np.stack(data_list, axis=-1)

# Final check for bad values in stack
if not np.isfinite(X_tensor_raw).all():
    n_bad = (~np.isfinite(X_tensor_raw)).sum()
    raise ValueError(f"X_tensor_raw has {n_bad} NaN/inf values")

# Ensure that X and y lengths match
assert X_tensor_raw.shape[0] == len(y_tensor_raw), "X and y row counts differ"

# Show channel names with index
print("==================")
print("Idx: Channel Name")
print("==================")
for i, name in enumerate(channel_names):
    print(f"{i}: {name}")

Idx: Channel Name
0: F
1: Np
2: P4
3: P5
4: P6
5: Tp
6: V
7: Vx
8: Xl
9: Xs


In [267]:
class_counts = np.bincount(y_tensor_raw)
print(f"Unique labels: {', '.join(df.iloc[:,0].unique().tolist())}")
print(f"Total NSEP:    {class_counts[0]}")
print(f"Total SEP:     {class_counts[1]}")
print()
print(f"X_tensor_raw shape: {X_tensor_raw.shape}")  
print(f"y_tensor_raw shape: {y_tensor_raw.shape}")
print()
print(f"Dataset range: {onset.min().date()} to {onset.max().date()}")

Unique labels: NSEP, gt100, gt10, gt60, gt30
Total NSEP:    17625
Total SEP:     169

X_tensor_raw shape: (17794, 288, 10)
y_tensor_raw shape: (17794,)

Dataset range: 1998-01-01 to 2013-12-31


# Chronological 70/10/20 Split

In [268]:
# Create boolean masks for each class
nsep_indices = (y_tensor_raw == 0)
sep_indices = (y_tensor_raw == 1)

# Separate X and y for SEP and non-SEP
X_nsep = X_tensor_raw[nsep_indices]
y_nsep = y_tensor_raw[nsep_indices]

# Separate X and y for SEP
X_sep = X_tensor_raw[sep_indices]
y_sep = y_tensor_raw[sep_indices] 

# Split Non-SEP data chronologically into Train/Val (80%) and Test (20%)
X_nsep_tv, X_nsep_test, y_nsep_tv, y_nsep_test = train_test_split(
    X_nsep, y_nsep, test_size=0.20, random_state=RAND_STATE, shuffle=False
)

# Split the 80% block into Train (70%) and Val (10%) (of total dataset)
X_nsep_train, X_nsep_val, y_nsep_train, y_nsep_val = train_test_split(
    X_nsep_tv, y_nsep_tv, test_size=0.125, random_state=RAND_STATE, shuffle=False
)

# Split SEP data chronologically into Train/Val (80%) and Test (20%)
X_sep_tv, X_sep_test, y_sep_tv, y_sep_test = train_test_split(
    X_sep, y_sep, test_size=0.20, random_state=RAND_STATE, shuffle=False
)

# Split the 80% block into Train (70%) and Val (10%) (of total dataset)
X_sep_train, X_sep_val, y_sep_train, y_sep_val = train_test_split(
    X_sep_tv, y_sep_tv, test_size=0.125, random_state=RAND_STATE, shuffle=False
)

# Combine non-SEP and SEP partitions
X_train_raw = np.concatenate([X_nsep_train, X_sep_train], axis=0)
y_train = np.concatenate([y_nsep_train, y_sep_train], axis=0)

X_val_raw = np.concatenate([X_nsep_val,   X_sep_val],   axis=0)
y_val = np.concatenate([y_nsep_val,   y_sep_val],   axis=0)

X_test_raw = np.concatenate([X_nsep_test,  X_sep_test],  axis=0)
y_test = np.concatenate([y_nsep_test,  y_sep_test],  axis=0)

#  Shuffle each set independently (randomizes sample order within the set without mixing time partitions)
X_train_raw, y_train = shuffle(X_train_raw, y_train, random_state=RAND_STATE)
X_val_raw, y_val = shuffle(X_val_raw,   y_val,   random_state=RAND_STATE)
X_test_raw, y_test = shuffle(X_test_raw,  y_test,  random_state=RAND_STATE)


print("================")
print(f"PRE-SPLIT")
print("================")
print(f"NSEP shape: {X_nsep.shape}")
print(f"SEP shape:  {X_sep.shape}")
print(f"Ratio:      {X_nsep.shape[0] / X_sep.shape[0]:.2f}:1")
print()

for name, X, y in [("TRAINING", X_train_raw, y_train),
                   ("VALIDATION", X_val_raw, y_val),
                   ("TEST", X_test_raw, y_test)]:
    n_sep = int((y == 1).sum())
    n_nsep = int((y == 0).sum())

    print("================")
    print(f"{name} DATA")
    print("================")
    print(f"X shape:    {X.shape}")
    print(f"y shape:    {y.shape}")
    print(f"NSEP count: {n_nsep}")
    print(f"SEP count:  {n_sep}")
    print(f"Ratio:      {n_nsep / n_sep:.2f}:1")
    print()

PRE-SPLIT
NSEP shape: (17625, 288, 10)
SEP shape:  (169, 288, 10)
Ratio:      104.29:1

TRAINING DATA
X shape:    (12455, 288, 10)
y shape:    (12455,)
NSEP count: 12337
SEP count:  118
Ratio:      104.55:1

VALIDATION DATA
X shape:    (1780, 288, 10)
y shape:    (1780,)
NSEP count: 1763
SEP count:  17
Ratio:      103.71:1

TEST DATA
X shape:    (3559, 288, 10)
y shape:    (3559,)
NSEP count: 3525
SEP count:  34
Ratio:      103.68:1



### Date Split Check

In [269]:
# Load in only the onset col from CSVs
onset_df = pd.read_csv(csv_files[0], usecols=[2])
onset = pd.to_datetime(onset_df.iloc[:, 0])
idx = np.arange(len(y_tensor_raw))

def split_idx(cls_idx):
    temp, test = train_test_split(cls_idx, test_size=0.20, shuffle=False)
    train, val = train_test_split(temp, test_size=0.125, shuffle=False)
    return {"train": train, "val": val, "test": test}

splits = {0: split_idx(idx[y_tensor_raw == 0]),
          1: split_idx(idx[y_tensor_raw == 1])}


print(f"{'Split':<7} {'Class':<5} {'Samples':<8} {'Begin':<12} {'End':<12}")
print("-" * 46)
for split in ["train", "val", "test"]:
    for cls, name in [(0, "NSEP"), (1, "SEP")]:
        d = onset.iloc[splits[cls][split]]
        begin = d.min().strftime('%Y-%m-%d')
        end   = d.max().strftime('%Y-%m-%d')
        print(f"{split:<7} {name:<5} {len(d):<8} {begin:<12} {end:<12}")

Split   Class Samples  Begin        End         
----------------------------------------------
train   NSEP  12337    1998-01-01   2004-10-21  
train   SEP   118      1998-04-20   2004-11-01  
val     NSEP  1763     2004-10-21   2011-07-31  
val     SEP   17       2004-11-07   2011-03-07  
test    NSEP  3525     2011-07-31   2013-12-31  
test    SEP   34       2011-03-08   2013-12-28  


# Pre-Processing

### Hybrid Normalization

In [270]:
z_score_only = ["V", "Vx"]

X_train_norm, X_val_norm, X_test_norm = [a.astype(np.float32) for a in (X_train_raw, X_val_raw, X_test_raw)]
splits = {'train': X_train_norm, 'validation': X_val_norm, 'test': X_test_norm}

norm_params = {}
clip_report = {'validation': {}, 'test': {}}  

# Loop through every channel
for j, name in enumerate(channel_names):
    # Fit training data
    values = X_train_norm[:, :, j].astype(np.float64)
    params = {} # Track normalization parameters 

    if name not in z_score_only:
        eps = np.percentile(values[values > 0], 0.1) # Get the smallest offset

        # Shift up if negative or zeros 
        min_value = values.min()
        shifted = bool(min_value + eps <= 0)
        if shifted:
            centred = values - min_value 
            delta = np.percentile(centred[centred > 0], 0.1)      
            eps = abs(min_value) + delta

        # Determine offset
        floor = (values + eps).min()

        # Apply log
        values = np.log10(np.maximum(values + eps, floor))

        params.update(method='log_zscore', eps=eps, floor=floor, shifted=shifted)

    elif name in z_score_only:
        params.update(method='zscore')

    else:
        raise ValueError(f"No normalization method defined for channel {j} ({name})")

    # Get parameters and save to dict
    params['mean'], params['std'] = values.mean(), values.std()    # train-only stats
    norm_params[name] = params

    # Apply normalization to all sets
    for split_name, X in splits.items():
        x = X[:, :, j].astype(np.float64)

        # Log normalization
        if name not in z_score_only:
            # Apply training offset
            n_clipped = int((x + eps < floor).sum())

            # Record any non zeros
            if n_clipped > 0 and split_name != 'train':
                clip_report[split_name][name] = n_clipped

            # Shift, clip, and take log
            x = np.log10(np.maximum(x + eps, floor))

        # Z score normalixation
        X[:, :, j] = (x - params['mean']) / params['std']

# Any NaNs get picked up here
for split_name, X in splits.items():
    assert np.isfinite(X).all(), f"{split_name} has NaN/Inf after normalization"

# Record any that required shifting
shifted_feats = [n for n, p in norm_params.items() if p.get('shifted')]
print("Features shifted to positive before log10:", shifted_feats or "None")


for split_name, rep in clip_report.items():
    if rep:
        total = splits[split_name].shape[0] * splits[split_name].shape[1]
        print(f"\n{total} {split_name} values clipped at log floor (below training range):")
        for n, c in rep.items():
            print(f"  {n}: {c} / {total} ({100.0 * c / total:.4f}%)")

print()
print("==========================")
print("HYBRID NORMALIZATION STATS")
print("==========================")
for name, params in norm_params.items():
    print(f"\n{name}:")
    for k, v in params.items():
        formatted_val = round(v,5) if isinstance(v, (float, np.floating)) else v
        print(f"  {k:<8}: {formatted_val}")

# Flatten 
X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
X_val_flat = X_val_norm.reshape(X_val_norm.shape[0], -1)
X_test_flat = X_test_norm.reshape(X_test_norm.shape[0], -1)

Features shifted to positive before log10: None

512640 validation values clipped at log floor (below training range):
  Xl: 1740 / 512640 (0.3394%)
  Xs: 1111 / 512640 (0.2167%)

1024992 test values clipped at log floor (below training range):
  Xl: 1206 / 1024992 (0.1177%)
  Xs: 16 / 1024992 (0.0016%)

HYBRID NORMALIZATION STATS

F:
  method  : log_zscore
  eps     : 1.49
  floor   : 1.76
  shifted : False
  mean    : 1.04812
  std     : 0.64968

Np:
  method  : log_zscore
  eps     : 0.23
  floor   : 0.26
  shifted : False
  mean    : 0.68984
  std     : 0.29806

P4:
  method  : log_zscore
  eps     : 0.00281
  floor   : 0.00281
  shifted : False
  mean    : -1.58292
  std     : 0.54222

P5:
  method  : log_zscore
  eps     : 0.00336
  floor   : 0.00336
  shifted : False
  mean    : -1.82731
  std     : 0.30069

P6:
  method  : log_zscore
  eps     : 0.00138
  floor   : 0.00196
  shifted : False
  mean    : -2.3274
  std     : 0.16304

Tp:
  method  : log_zscore
  eps     : 6005.0
 

# Evaluation Metrics

In [271]:
def true_skill_statistic(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    # Guard against divide by zero
    if (tp + fn == 0 or
        tn + fp == 0):
        return 0.0

    tss = tp / (tp + fn) - fp / (tn + fp)
    
    return float(tss)

def heidke_skill_score(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()

    numerator = 2 * ((tp * tn) - (fp * fn))
    denominator = ((tp + fn) * (fn + tn)) + ((tn + fp) * (tp + fp))

    # Guard against divide by zero
    if denominator == 0:
        return 0.0
        
    return float(numerator / denominator)

def brier_skill_score(y_true, y_prob):
    base_rate = np.mean(y_true)
    brier_ref = brier_score_loss(y_true, np.full(len(y_true), base_rate))
    return 1 - brier_score_loss(y_true, y_prob) / brier_ref

def evaluate_model_metrics( 
        y_actual, 
        y_pred, 
        y_prob, 
        model_name="Model"
    ):    
    conf_matrix = confusion_matrix(y_actual, y_pred)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_actual, y_pred),
        "Precision": precision_score(y_actual, y_pred, zero_division=0),
        "Recall": recall_score(y_actual, y_pred, zero_division=0),
        "F1-Score": f1_score(y_actual, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_actual, y_prob),
        "PR-AUC": average_precision_score(y_actual, y_prob),
        "Brier": brier_score_loss(y_actual, y_prob),
        "BSS" : brier_skill_score(y_actual, y_prob),
        "TSS": true_skill_statistic(conf_matrix),
        "HSS2": heidke_skill_score(conf_matrix),
        "Confusion Matrix": conf_matrix
    }
    
    return metrics

def print_metrics(metrics):
    print("===================")
    print("PERFORMANCE METRICS")
    print("===================")
    for name, val in metrics.items():
        formatted_val = round(val,5) if isinstance(val, (float, np.floating)) else val   
        print(f"{name:<17}: {formatted_val}")
    print()

# Baseline Classifications

### Logistic Regression

In [76]:
def best_tss_threshold(y_true, y_prob):
    fpr, tpr, threshold = roc_curve(y_true, y_prob)
    tss = tpr - fpr
    i = np.argmax(tss)
    return threshold[i], tss[i]

# Grid search strategy on different C and class weights to see what provides best tss on validation set
results = []
for C in [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]:
    for class_weight in ['balanced', {0: 1, 1: 10}, {0: 1, 1: 30}]:
        model = LogisticRegression(
            C=C, 
            class_weight=class_weight, 
            max_iter=2000,
            random_state=RAND_STATE
        )

        # Train model with new parameters
        model.fit(X_train_flat, y_train)
        prob = model.predict_proba(X_val_flat)[:, 1]

        # Find which threshold produces best TSS
        threshold, tss = best_tss_threshold(y_val, prob)
        results.append((tss, C, class_weight, threshold))
        print(f"C={C:<7} class_weight={str(class_weight):<15} TSS={tss:.4f}  threshold={threshold:.5f}")

# Find what test produced the best TSS result on validation set
best_tss, best_C, best_class_weight, best_threshold = max(results, key=lambda r: r[0])
print(f"\nBest: C={best_C}, class_weight={best_class_weight}, threshold={best_threshold:.4f}, TSS={best_tss:.5f}\n")

# Run on validation set
logreg_model = LogisticRegression(
    C=best_C,                    
    class_weight=best_class_weight,        
    max_iter=2000,              
    random_state=RAND_STATE
)

logreg_model.fit(X_train_flat, y_train)

# Make predictions using tuned best threshold
y_val_prob = logreg_model.predict_proba(X_val_flat)[:, 1]
y_val_pred = (y_val_prob >= best_threshold).astype(int)

logreg_results = evaluate_model_metrics(
    y_actual=y_val,
    y_pred=y_val_pred,
    y_prob=y_val_prob,
    model_name="Logistic Regression (Val)"
)
print_metrics(logreg_results)


C=1e-06   class_weight=balanced        TSS=0.6320  threshold=0.48168
C=1e-06   class_weight={0: 1, 1: 10}   TSS=0.6117  threshold=0.09631
C=1e-06   class_weight={0: 1, 1: 30}   TSS=0.6303  threshold=0.21956
C=1e-05   class_weight=balanced        TSS=0.7695  threshold=0.48182
C=1e-05   class_weight={0: 1, 1: 10}   TSS=0.6620  threshold=0.09565
C=1e-05   class_weight={0: 1, 1: 30}   TSS=0.7203  threshold=0.22957
C=0.0001  class_weight=balanced        TSS=0.5788  threshold=0.47323
C=0.0001  class_weight={0: 1, 1: 10}   TSS=0.6795  threshold=0.08538
C=0.0001  class_weight={0: 1, 1: 30}   TSS=0.5811  threshold=0.22565
C=0.001   class_weight=balanced        TSS=0.4822  threshold=0.41412
C=0.001   class_weight={0: 1, 1: 10}   TSS=0.4737  threshold=0.09973
C=0.001   class_weight={0: 1, 1: 30}   TSS=0.4776  threshold=0.20141
C=0.01    class_weight=balanced        TSS=0.4572  threshold=0.13391
C=0.01    class_weight={0: 1, 1: 10}   TSS=0.4402  threshold=0.04288
C=0.01    class_weight={0: 1, 1: 3

In [77]:
# Run on test set using tuned parameters
y_test_prob = logreg_model.predict_proba(X_test_flat)[:, 1]
y_test_pred = (y_test_prob >= best_threshold).astype(int) 

logreg_test_results = evaluate_model_metrics(
    y_actual=y_test,
    y_pred=y_test_pred,
    y_prob=y_test_prob,
    model_name="Logistic Regression (Test)"
)
print_metrics(logreg_test_results)

PERFORMANCE METRICS
Model            : Logistic Regression (Test)
Accuracy         : 0.86766
Precision        : 0.04
Recall           : 0.55882
F1-Score         : 0.07466
ROC-AUC          : 0.73727
PR-AUC           : 0.06977
Brier            : 0.14836
BSS              : -14.67952
TSS              : 0.42946
HSS2             : 0.05786
Confusion Matrix : [[3069  456]
 [  15   19]]



### XGBoost

In [ ]:
rng = random.Random(RAND_STATE) # Define rng here so random state in build_model doesn't reset other random choices
K = 30 # Number of random sampling

learn_rate_opts = [0.1, 0.05, 0.02]
spw_opts = [1, 10, 30, 104]
depth_opts = [1, 2, 3, 4]
mcw_opts = [1, 3, 5, 10, 20]
cbt_opts = [0.1, 0.3, 0.5]

# Get random sample using grid to avoid duplicates
grid = list(product(learn_rate_opts, spw_opts, depth_opts, mcw_opts, cbt_opts))
sampled = rng.sample(grid, K)

results_xgb = []
for learn_rate, scale_pos_weight, max_depth, min_child_weight, colsample_bytree in sampled:

    model = XGBClassifier(
        scale_pos_weight=scale_pos_weight,  # Weights class imbalance
        max_depth=max_depth,                # Decision tree max depth
        min_child_weight=min_child_weight,  # Min sum of weight a leaf must have to exist
        n_estimators=1000,                  # No. of sequential decision trees (upper bound)
        learning_rate=learn_rate,
        subsample=0.7,                      # Random subsample tree is trained on
        colsample_bytree=colsample_bytree,  # Random percent of cols each tree sees
        reg_lambda=10,                      # L2 penalty on leaf values
        eval_metric="aucpr",
        early_stopping_rounds=50,           # Early stopping condition
        random_state=RAND_STATE,
        n_jobs=-1,
    )

    # Train model with new parameters
    model.fit(
        X_train_flat, y_train,
        eval_set=[(X_val_flat, y_val)],
        verbose=False
    )

    # Find which threshold produces best TSS
    prob = model.predict_proba(X_val_flat)[:, 1]
    thr, tss = best_tss_threshold(y_val, prob)
    results_xgb.append((tss, learn_rate, scale_pos_weight, max_depth, min_child_weight, colsample_bytree, thr, model.best_iteration))
    print(f"lr={learn_rate:<5} spw={scale_pos_weight:<4} md={max_depth} mcw={min_child_weight} cbt={colsample_bytree} trees={model.best_iteration:<4} val TSS={tss:.5f}  thr={thr:.5f}")

# Find what test produced the best TSS result on validation set
(best_tss, best_lr, best_spw, best_md, best_mcw, best_cbt, best_thr_xgb, best_iter) = max(results_xgb, key=lambda r: r[0])
print(f"\nBest: lr={best_lr}, spw={best_spw}, md={best_md}, mcw={best_mcw}, cbt={best_cbt}, trees={best_iter + 1}, thr={best_thr_xgb:.5f}, val TSS={best_tss:.5f}\n")

# Run on validation set
xgb_model = XGBClassifier(
    scale_pos_weight=best_spw,
    max_depth=best_md,
    min_child_weight=best_mcw,
    n_estimators=best_iter + 1,        
    learning_rate=best_lr,
    subsample=0.7,
    colsample_bytree=best_cbt,
    reg_lambda=10,
    random_state=RAND_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train_flat, y_train)

# Make predictions using tuned best threshold
y_val_prob_xgb = xgb_model.predict_proba(X_val_flat)[:, 1]
y_val_pred_xgb = (y_val_prob_xgb >= best_thr_xgb).astype(int)

xgb_results = evaluate_model_metrics(
    y_actual=y_val,
    y_pred=y_val_pred_xgb,
    y_prob=y_val_prob_xgb,
    model_name="XGBoost (Val)"
)
print_metrics(xgb_results)

In [ ]:
# Run on test set using tuned parameters
y_test_prob_xgb = xgb_model.predict_proba(X_test_flat)[:, 1]
y_test_pred_xgb = (y_test_prob_xgb >= best_thr_xgb).astype(int)

xgb_test_results = evaluate_model_metrics(
    y_actual=y_test,
    y_pred=y_test_pred_xgb,
    y_prob=y_test_prob_xgb,
    model_name="XGBoost (Test)"
)
print_metrics(xgb_test_results)

### GRU

In [ ]:
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# GRU model───────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()

        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]         
        out = self.bn(out)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out).squeeze(-1)


# GRU hyperparameter grid─────────────────────────────────
GRU_GRID = [
    # Small models
    {"units": 32,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 32,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Medium models
    {"units": 64,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 64,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Larger single-layer models
    {"units": 128, "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 32},

    # Two-layer models
    {"units": 64,  "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.001,  "batch_size": 32},

    # Lower learning rate variants
    {"units": 64,  "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 32},
]

def gru_label(p):
    return (
        f"units={p['units']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )

# Train GRU and evaluate on a provided test split──────────
def train_and_evaluate_gru(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5,
    seed=RAND_STATE
):
    torch.manual_seed(seed)
    
    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model
    input_size = X_train.shape[2]

    model = GRUModel(
        input_size=input_size,
        hidden_size=params["units"],
        num_layers=params["layers"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            if len(idx) < 2:                        
                continue    

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Tune threshold on val                                                
    model.eval()
    with torch.no_grad():                                                  
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()              
    thr, _ = best_tss_threshold(y_val.astype(int), y_val_prob)             

    # Prediction on the eval split
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= thr).astype(int) 
    infer_time = time.time() - t0

    metrics = evaluate_model_metrics(
        y_actual=y_test.astype(int),                
        y_pred=y_pred,
        y_prob=y_prob,   
        model_name="GRU"
    )
    metrics["threshold"] = thr
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

In [ ]:
# ══════════════════════════════════════════════════════════════
# RUN GRU EXPERIMENTS
# Tune on validation set, then evaluate best params on test set
# ══════════════════════════════════════════════════════════════
N_FINAL_RUNS = 2

print(f"{'═' * 60}")
print("  Classifier : GRU")
print(f"{'═' * 60}")

# Sanity check: GRU expects (samples, 288, 10)
print("X_train:", X_train_norm.shape, " X_val:", X_val_norm.shape, " X_test:", X_test_norm.shape)

class_ratio = round((y_train == 0).sum() / (y_train == 1).sum(), 1)
print(f"Train class ratio 0:1 = {class_ratio}:1\n")

# Step 1: Hyperparameter search on validation set only───────────────────
print(f"Searching {len(GRU_GRID)} GRU combinations on validation set...\n")

best_params = None
best_tss = -999
best_val_metrics = None

for i, params in enumerate(GRU_GRID):
    print(f"[{i + 1}/{len(GRU_GRID)}] {gru_label(params)}", flush=True)

    # Validation set is used as the evaluation split during tuning
    metrics, _ = train_and_evaluate_gru(
        params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_val_norm,   y_val,
        class_ratio=class_ratio,
        seed=RAND_STATE
    )

    print(
        f"    Val TSS={metrics['TSS']:.4f}  "         
        f"F1={metrics['F1-Score']:.4f}  "             
        f"Recall={metrics['Recall']:.4f}  "           
        f"Thr={metrics['threshold']:.4f}  "           
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["TSS"] > best_tss:
        best_tss = metrics["TSS"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics

print("Best GRU hyperparameters selected from validation set:")
print(f"  Params  : {gru_label(best_params)}")
print_metrics(best_val_metrics) 

# Step 2: Final runs on the test set with the best hyperparameters───────
print(f"\nRunning {N_FINAL_RUNS} final runs with the best hyperparameters...\n")
print("─" * 70)
print(f"Params  : {gru_label(best_params)}")
print(f"Ratio   : {class_ratio}:1")
print("─" * 70)

metrics_list = []

for run in range(N_FINAL_RUNS):
    print(f"Run {run + 1}...", flush=True)

    metrics, _ = train_and_evaluate_gru(
        best_params,
        X_train_norm, y_train,
        X_val_norm,   y_val,
        X_test_norm,  y_test,
        class_ratio=class_ratio,
        seed=RAND_STATE + run # Make sure the runs are different
    )

    metrics_list.append(metrics)

    print_metrics(metrics)

tss_runs = [m["TSS"] for m in metrics_list]        
print(f"Test TSS over {N_FINAL_RUNS} runs: mean={np.mean(tss_runs):.4f}  std={np.std(tss_runs):.4f}")  

print("\nGRU experiment complete.")

### SVM 

In [219]:
# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION (hybrid-normalized data only)
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    return Catch22(catch24=False).fit_transform(X_chunk).to_numpy()

def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 200) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    # Safety net: replace any non-finite values with the channel's median
    if not np.isfinite(X).all():
        print(f"    ⚠️  non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]                       # view, so edits land in X
            bad = ~np.isfinite(col)
            col[bad] = np.median(col[~bad])

    # sktime expects (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(delayed(_catch22_chunk)(ch) for ch in chunks)
    X_feat = np.vstack(results)

    assert X_feat.shape == (n_samples, 22 * n_channels), f"Unexpected shape: {X_feat.shape}"
    return X_feat


# Extract features for each split──────────────────────────
print("Extracting Catch22 features …")
X_train_c22 = extract_catch22(X_train_norm, label="train")
X_val_c22   = extract_catch22(X_val_norm,   label="val")
X_test_c22  = extract_catch22(X_test_norm,  label="test")
print(f"train {X_train_c22.shape}  val {X_val_c22.shape}  test {X_test_c22.shape}")


# ══════════════════════════════════════════════════════════════
# IMPUTE — fit on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

n_bad = [int((~np.isfinite(a)).sum()) for a in (X_train_c22, X_val_c22, X_test_c22)]

if any(n_bad):
    print(f"non-finite catch22 values — train: {n_bad[0]}  val: {n_bad[1]}  test: {n_bad[2]} → imputing")

    # Treat inf as missing too, so the imputer handles both
    X_train_c22, X_val_c22, X_test_c22 = [
        np.where(np.isfinite(a), a, np.nan) for a in (X_train_c22, X_val_c22, X_test_c22)
    ]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imp = SimpleImputer(strategy="median", keep_empty_features=True)
        X_train_c22 = imp.fit_transform(X_train_c22)
        X_val_c22   = imp.transform(X_val_c22)
        X_test_c22  = imp.transform(X_test_c22)
else:
    print("✓ No NaN or Inf in catch22 features — no imputation needed")

X_train_c22, X_val_c22, X_test_c22 = [
    a.astype(np.float32) for a in (X_train_c22, X_val_c22, X_test_c22)
]

print(f"Ready: train {X_train_c22.shape}  val {X_val_c22.shape}  test {X_test_c22.shape}")


CLIP_Z = 10
X_train_c22, X_val_c22, X_test_c22 = [
    np.clip(a, -CLIP_Z, CLIP_Z).astype(np.float32)
    for a in (X_train_c22, X_val_c22, X_test_c22)
]

print(f"train col mean range: {X_train_c22.mean(0).min():.2g} to {X_train_c22.mean(0).max():.2g}")
print(f"train col std  range: {X_train_c22.std(0).min():.2g} to {X_train_c22.std(0).max():.2g}")
print(f"val   col max |z|   : {np.abs(X_val_c22).max():.1f}")

Extracting Catch22 features …
    train — 12455 samples in 63 chunks …
    val — 1780 samples in 9 chunks …
    test — 3559 samples in 18 chunks …
train (12455, 220)  val (1780, 220)  test (3559, 220)
non-finite catch22 values — train: 84320  val: 20276  test: 33898 → imputing
Ready: train (12455, 220)  val (1780, 220)  test (3559, 220)
train col mean range: -0.35 to 10
train col std  range: 0 to 3.8
val   col max |z|   : 10.0


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# SVM HYPERPARAMETER GRID
# Tune only on hybrid validation set
# ══════════════════════════════════════════════════════════════

SVM_GRID = [
    {"model": LinearSVC, "params": {"C": 0.001, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.01, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.05, "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.1,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.5,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 1.0,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 3.0,  "class_weight": "balanced", "max_iter": 10000}},
    {"model": LinearSVC, "params": {"C": 0.01,  "class_weight": {0: 1, 1: 10}, "max_iter": 10000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.01,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 1.0,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": 0.001,   "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": {0: 1, 1: 10}, "cache_size": 2000}}
]


def combo_label(combo):
    ModelClass = combo["model"]
    params = combo["params"]

    cw = "balanced" if params.get("class_weight") == "balanced" else "none"

    if ModelClass == LinearSVC:
        return f"LinearSVC | C={params['C']} | cw={cw}"
    else:
        return f"RBF-SVC   | C={params['C']} | gamma={params['gamma']} | cw={cw}"


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON HYBRID VALIDATION SET ONLY
# ══════════════════════════════════════════════════════════════

print("═" * 70)
print("SVM hyperparameter search on HYBRID validation set only")
print("═" * 70)

d_hybrid = {
    "X_train": X_train_c22, "y_train": y_train,
    "X_val":   X_val_c22,   "y_val":   y_val,
    "X_test":  X_test_c22,  "y_test":  y_test,
}

best_tss = -np.inf
best_model_class = None
best_params = None
best_val_metrics = None
best_model = None

for combo in SVM_GRID:
    ModelClass = combo["model"]
    params = copy.deepcopy(combo["params"])

    model = ModelClass(**params)

    t0 = time.time()
    model.fit(d_hybrid["X_train"], d_hybrid["y_train"])
    y_val_pred = model.predict(d_hybrid["X_val"])
    y_val_prob = expit(model.decision_function(d_hybrid["X_val"]))
    elapsed = time.time() - t0

    val_metrics = evaluate_model_metrics(       
        y_actual=d_hybrid["y_val"].astype(int), 
        y_pred=y_val_pred,                      
        y_prob=y_val_prob,                      
        model_name=combo_label(combo)           
    )

    print(
        f"{combo_label(combo):<55} "
        f"Val TSS={val_metrics['TSS']:.4f} | "
        f"F1={val_metrics['F1-Score']:.4f} | "
        f"{elapsed:.1f}s"
    )

    if val_metrics["TSS"] > best_tss:
        best_tss = val_metrics["TSS"]
        best_model_class = ModelClass
        best_params = params
        best_val_metrics = val_metrics
        best_model = model

print("\nBest hyperparameters selected from hybrid validation set:")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")
print_metrics(best_val_metrics)


# ══════════════════════════════════════════════════════════════
# STEP 2: ONE-TIME TEST EVALUATION WITH THE BEST MODEL
# ══════════════════════════════════════════════════════════════

y_test_pred = best_model.predict(d_hybrid["X_test"])                      
y_test_prob = expit(best_model.decision_function(d_hybrid["X_test"]))     

svm_test_results = evaluate_model_metrics(                 
    y_actual=d_hybrid["y_test"].astype(int),               
    y_pred=y_test_pred,                                    
    y_prob=y_test_prob,                                    
    model_name=f"{best_model_class.__name__} (test)"       
)
print_metrics(svm_test_results)                            

# Triplet Loss

### Loss Function

In [272]:
def triplet_loss(anchor, positive, negative, margin=1.0):
    d_ap = F.pairwise_distance(anchor, positive)   # distance from anchor to positive
    d_an = F.pairwise_distance(anchor, negative)   # distance from anchor to negative
    losses = F.relu(d_ap - d_an + margin)          # loss (0 is good. More positive the worse)
    return losses.mean()

### 1-D Convolutional Encoder

In [273]:
class CausalConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.0):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation                
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel_size, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation)
        self.act = nn.LeakyReLU(0.01)      
        self.drop = nn.Dropout1d(dropout)                         
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()  # remap C_in -> C_out so shapes match

    def forward(self, x):                       # x                                       
        out = F.pad(x, (self.left_pad, 0))      # pad zeros on the left                   
        out = self.act(self.conv1(out))         # conv1 (C_in -> C_out), then LeakyReLU   
        out = F.pad(out, (self.left_pad, 0))    # pad zeros on the left                   
        out = self.act(self.conv2(out))         # conv2 (C_out -> C_out), then LeakyReLU  
        out = self.drop(out)                    # Dropout some of the neurons in training 
        return out + self.skip(x)               # changes + original                      

    
class TimeSeriesEncoder(nn.Module):
    def __init__(self, in_ch, hidden_ch=32, depth=6, emb_dim=64, kernel_size=3, normalize=True, dropout=0.0):
        super().__init__()
        blocks = []
        for i in range(depth):
            blocks.append(
                CausalConvBlock(
                    in_ch if i == 0 else hidden_ch,  
                    hidden_ch,
                    kernel_size,
                    dilation=2 ** i,             
                    dropout=dropout,
                )
            )               
        self.blocks = nn.Sequential(*blocks)      
        self.linear = nn.Linear(2 * hidden_ch, emb_dim)
        self.normalize = normalize

    def forward(self, out):                     # x                             
        out = out.permute(0, 2, 1)              # swap channels and time        
        out = self.blocks(out)                  # lean C traces                 
        peaks = out.max(dim=2).values           # keep just peaks               
        latest = out[:, :, -1]                  # features at the final timestep
        out = torch.cat([peaks, latest], dim=1) # peaks + latest                
        out = self.linear(out)                  # remix to fingerprint          
        if self.normalize:
            out = F.normalize(out, dim=1)       # each row gets length 1        
        return out

### GRU Encoder

In [274]:
class GRUEncoder(nn.Module):
    def __init__(self, in_ch, hidden_ch=32, depth=2, emb_dim=64, normalize=True, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(in_ch, hidden_ch, num_layers=depth, batch_first=True, dropout=dropout if depth > 1 else 0.0)
        self.linear = nn.Linear(hidden_ch, emb_dim)
        self.normalize = normalize

    def forward(self, x):        
        _, h = self.gru(x)       
        out = self.linear(h[-1]) 
        return F.normalize(out, dim=1) if self.normalize else out

### Embedding Extraction

In [301]:
@torch.no_grad()
def embed(enc, X, batch_size=512):
    enc.eval()
    # Run through in sizes of batch_size at a time
    chunks = [enc(X[i:i + batch_size]) for i in range(0, len(X), batch_size)]
    return torch.cat(chunks).cpu().numpy()

### Triplet Mining

In [302]:
def pairwise_dist(A, B):
    # Euclidean distance between every row of A (n, d) and every row of B (m, d) -> (n, m).
    a2 = (A ** 2).sum(axis=1)[:, None]
    b2 = (B ** 2).sum(axis=1)[None, :]
    d2 = np.maximum(a2 + b2 - 2.0 * (A @ B.T), 0.0)
    return np.sqrt(d2)

@torch.no_grad()
def refresh_hard_pool(enc, X_train_t, y, pool):
    # NSEPs currently closest to any SEP in the encoder's embedding space (closest first).
    y = np.asarray(y)
    sep_idx, nsep_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    Z = embed(enc, X_train_t)
    enc.train()                                                   
    d_to_sep = pairwise_dist(Z[nsep_idx], Z[sep_idx]).min(axis=1) 
    return nsep_idx[np.argsort(d_to_sep)[:pool]]

def sample_batch_indices(y, batch_size, rng,
                         sep_fraction=0.5,      # share of the batch that is SEP anchors
                         hard_frac=0.0,         # share of the NSEP slots filled from hard_pool
                         hard_pool=None,        # NSEP row numbers closest to the SEPs (from refresh_hard_pool)
                        ):
    y = np.asarray(y)

    # Separate dataset indices into SEP and NSEP pools
    sep_idx, nsep_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)

    # Compute sample size for batches
    n_sep = int(round(batch_size * sep_fraction))
    n_nsep = batch_size - n_sep
    n_hard = int(round(n_nsep * hard_frac)) if hard_pool is not None else 0

    # Sample SEP indices
    sep = rng.choice(sep_idx, n_sep, replace=n_sep > len(sep_idx))

    # Sample hard NSEP indices from the hard pool (if active)
    hard_nsep = (
        rng.choice(hard_pool, n_hard, replace=n_hard > len(hard_pool))
        if n_hard
        else np.empty(0, dtype=int)
    )

    # Sample the remaining majority slots from standard NSEPs (excluding hard ones to avoid duplicates)
    rest_pool = np.setdiff1d(nsep_idx, hard_nsep)  
    rest = rng.choice(
        rest_pool,
        n_nsep - n_hard,
        replace=(n_nsep - n_hard) > len(rest_pool),
    )

    # Combine all components
    return np.concatenate([sep, hard_nsep, rest])

@torch.no_grad()
def mine_semi_hard(emb, y, margin=0.3, include_hard=False, max_triplets=4096):
    # Compute pairwise Euclidean distance matrix for all embeddings in the batch
    d = torch.cdist(emb, emb)

    # Make a 3D pairwise distance tensor
    d_ap, d_an = d.unsqueeze(2), d.unsqueeze(1)

    # Create boolean masks to filter valid triplet combinations
    same = y[:, None] == y[None, :]  # True where labels match
    eye = torch.eye(len(y), dtype=torch.bool, device=y.device) 
    valid = (same & ~eye).unsqueeze(2) & (~same).unsqueeze(1)

    # Apply triplet mining boundary conditions based on margin
    # Semi-hard condition: negative is further than positive, but within the margin window
    mask = valid & (d_an < d_ap + margin)
    if not include_hard:
        mask &= (d_an > d_ap)

    # Extract indices of valid triplets
    a, p, n = mask.nonzero(as_tuple=True)

    # Check if the mined triplet count exceeds the safety cap
    if len(a) > max_triplets:
        keep = torch.randperm(len(a), device=a.device)[:max_triplets]
        a, p, n = a[keep], p[keep], n[keep]

    return a, p, n

### Classifier Heads

In [277]:
class Model:
    def fit(self, Z, y):
        self._fit(Z, y)
        raw = self._raw_train(Z)                       
        self._mu, self._sd = raw.mean(), raw.std() + 1e-12
        return self
    def score(self, Z): 
        return expit((self._raw(Z) - self._mu) / self._sd)
    def score_train(self, Z): 
        return expit((self._raw_train(Z) - self._mu) / self._sd)
    def _raw_train(self, Z):  
        return self._raw(Z)      

class LogRegModel(Model):
    def __init__(self, C=1.0, class_weight="balanced"):
        lr = LogisticRegression(C=C, class_weight=class_weight, max_iter=2000)
        self.model = lr
    def _fit(self, Z, y):     
        self.model.fit(Z, y)
    def _raw(self, Z):        
        return self.model.decision_function(Z)
    def score(self, Z):       
        return self.model.predict_proba(Z)[:, 1]   
    def score_train(self, Z): 
        return self.score(Z)

class CentroidModel(Model):
    def __init__(self, use_nsep=True):
        self.use_nsep = use_nsep
    def _fit(self, Z, y):
        self.mu_sep  = Z[y == 1].mean(axis=0)
        self.mu_nsep = Z[y == 0].mean(axis=0)
    def _raw(self, Z):
        d_sep = np.linalg.norm(Z - self.mu_sep, axis=1)
        if not self.use_nsep:
            return -d_sep
        return np.linalg.norm(Z - self.mu_nsep, axis=1) - d_sep

class KnnModel(Model):
    def __init__(self, k=5):
        self.k = k
    def _fit(self, Z, y):
        self.sep_pos = np.flatnonzero(y == 1)
        self.Z_sep = Z[self.sep_pos]
    def _raw(self, Z):
        d = pairwise_dist(Z, self.Z_sep)
        return -np.sort(d, axis=1)[:, :self.k].mean(axis=1)
    def _raw_train(self, Z):
        d = pairwise_dist(Z, self.Z_sep)
        d[self.sep_pos, np.arange(len(self.sep_pos))] = np.inf 
        return -np.sort(d, axis=1)[:, :self.k].mean(axis=1)

class SvmModel(Model):
    def __init__(self, kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"):
        self.model = SVC(kernel=kernel, C=C, gamma=gamma, class_weight=class_weight, cache_size=2000)
    def _fit(self, Z, y):  
        self.model.fit(Z, y)
    def _raw(self, Z):     
        return self.model.decision_function(Z)

HEADS = {"logreg": LogRegModel, "centroid": CentroidModel, "knn": KnnModel, "svm": SvmModel}

def make_model(name, params=None):
    return HEADS[name](**(params or {}))

### Classifier Evaluation

In [278]:
def evaluate_head(head, head_params, Z_train, y_train, Z_val, y_val, Z_eval=None, y_eval=None, model_name="Model"):
    # Instantiate the model using factory function and fit on training embeddings
    clf = make_model(head, head_params).fit(Z_train, y_train)

    # Generate score predictions for training and validation sets
    s_train, s_val = clf.score_train(Z_train), clf.score(Z_val)

    # Find best threshold
    fpr, tpr, thr = roc_curve(y_val, s_val)

    # Maximize for TSS
    threshold = float(thr[np.argmax(tpr - fpr)])

    # Select the evaluation target (defaults to validation set if no test set is provided)
    if Z_eval is None:
        y_eval, s_eval = y_val, s_val
    else:
        s_eval = clf.score(Z_eval)

    # Evaluate
    y_pred = (s_eval >= threshold).astype(int)
    metrics = evaluate_model_metrics(y_eval, y_pred, s_eval, model_name=model_name)

    # Compute extra training metrics
    fpr_t, tpr_t, _ = roc_curve(y_train, s_train)
    metrics["train_ROC_AUC"] = roc_auc_score(y_train, s_train)
    metrics["train_TSS"] = float(np.max(tpr_t - fpr_t))
    metrics["Threshold"] = threshold

    return metrics

### Encoder Training

In [279]:
def evaluate_step(step, train_loss, enc, X_train_t, y_train, X_val_t, y_val, head, head_params):
    Z_train, Z_val = embed(enc, X_train_t), embed(enc, X_val_t)
    results = evaluate_head(head, head_params, Z_train, y_train, Z_val, y_val)
    enc.train()
    return {"step": step, "train_loss": train_loss, "TSS": results["TSS"], "train_TSS": results["train_TSS"], "ROC_AUC": results["ROC-AUC"], "train_ROC_AUC": results["train_ROC_AUC"]}

def print_step_metrics(metrics):
    print(f"step {metrics['step']:5d} | loss {metrics['train_loss']:.4f} | Train TSS {metrics['train_TSS']:.3f} | Val TSS {metrics['TSS']:.3f} | Train ROC-AUC {metrics['train_ROC_AUC']:.3f} | Val ROC-AUC {metrics['ROC_AUC']:.3f}")

In [303]:
def train_encoder(X_train_t, y_train, X_val_t, y_val, 
                  seed=RAND_STATE,          # Random state seed
                  n_steps=1000,             # Number of training steps
                  lr=1e-3,                  # Learning rate
                  batch_size=256,           # Number of anchors per step (triplets are mined inside the batch)
                  hidden_ch=32,             # Number of pattern detectors
                  emb_dim=32,               # Embedding dimension
                  weight_decay=0.01,         # L2 penalty: shrinks weights to 0
                  depth=6,                  # Number of stacked blocks
                  sep_fraction=0.5,         # Portion of each batch that is SEP
                  hard_frac=0.5,            # Portion of the NSEP slots drawn from the hard pool (0 = fully random batches)
                  hard_pool=2000,           # Hard pool = this many NSEPs closest to any SEP in embedding space
                  refresh_every=250,        # Recompute the hard pool from the current encoder every this many steps
                  include_hard=False,       # Also use negatives closer than the positive (not only semi-hard)
                  max_triplets=15000,       # Cap on mined triplets per step
                  margin=0.3,               # How much farther negative must be
                  eval_every=100,           # How often to evaluate 
                  dropout=0.0,              # Training dropout
                  head="logreg",            # which classifier the printed curves use
                  head_params=None,         # classifier's settings
                  encoder="cnn"             # encoder selection
                ):

    device = X_train_t.device
    torch.manual_seed(seed)                 # controls the encoder's starting weights
    rng = np.random.default_rng(seed)       # controls which windows are sampled
    y_np = np.asarray(y_train)
    y_t = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # Initialize encoder and optimizer (encoder's weights)
    if (encoder == "cnn"):
        enc = TimeSeriesEncoder(in_ch=X_train_t.shape[2], hidden_ch=hidden_ch, depth=depth, emb_dim=emb_dim, dropout=dropout).to(device)
    else:
        enc = GRUEncoder(in_ch=X_train_t.shape[2], hidden_ch=hidden_ch, depth=depth, emb_dim=emb_dim, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(enc.parameters(), lr=lr, weight_decay=weight_decay)

    # Evaluate untrained model (step 0)
    history = [evaluate_step(0, float("nan"), enc, X_train_t, y_train, X_val_t, y_val, head, head_params)]
    print_step_metrics(history[0])

    pool_idx = None
    recent_losses = []                           
    for step in range(1, n_steps + 1):
        # Periodically rebuild the pool of NSEPs that currently look most like SEPs
        if hard_frac > 0 and (step - 1) % refresh_every == 0:
            pool_idx = refresh_hard_pool(enc, X_train_t, y_np, hard_pool)

        # Pick a batch and embed it
        idx = torch.as_tensor(sample_batch_indices(y_np, batch_size, rng, sep_fraction, hard_frac, pool_idx), dtype=torch.long)
        emb = enc(X_train_t[idx])

        # Mine triplets from the batch embeddings, then apply the same triplet loss as before
        a, p, n = mine_semi_hard(emb.detach(), y_t[idx], margin=margin, include_hard=include_hard, max_triplets=max_triplets)
        loss = triplet_loss(emb[a], emb[p], emb[n], margin=margin) if len(a) else emb.sum() * 0

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        recent_losses.append(loss.item())

        # Evaluate on validation every `eval_every` steps, and at the very end
        if step % eval_every == 0 or step == n_steps:
            history.append(evaluate_step(step, float(np.mean(recent_losses)), enc, X_train_t, y_train, X_val_t, y_val, head, head_params))
            recent_losses = []
            print_step_metrics(history[-1])

    return enc, history

## Experiments

### Experiment Runner

In [304]:
RESULTS = {} # Stores all run results for later comparison

In [305]:
def run_triplet(cfg, seed, X_eval_t, y_eval):
    # Train encoder on train, fit head on train, pick threshold on val, score on the eval set.
    enc, _ = train_encoder(X_train_t, y_train, X_val_t, y_val, seed=seed, **cfg)
    Z_train = embed(enc, X_train_t)
    Z_val = embed(enc, X_val_t)
    Z_eval = embed(enc, X_eval_t)
    return evaluate_head(cfg["head"], cfg["head_params"], Z_train, y_train, Z_val, y_val, Z_eval, y_eval, model_name=f"Triplet+{cfg['head']} seed {seed}")

def run_experiment(name, grid, n_search_seeds, n_final_runs, select_metric):
    print("═" * 70)
    print(f"{name}: searching {len(grid)} configs x {n_search_seeds} seeds on validation")
    print("═" * 70)

    # Loop through configs and seeds. Train and eval
    rows = []
    for c, cfg in enumerate(grid):
        for i in range(n_search_seeds):
            seed = RAND_STATE + i
            # Print run header
            header_label = f"\n[config {c+1}/{len(grid)} | seed {seed}] {cfg}"
            print(header_label)

            # Run training and eval
            t0 = time.perf_counter()      
            m = run_triplet(cfg, seed, X_val_t, y_val)
            secs = time.perf_counter() - t0 

            # Add to df and print results
            rows.append({"config": c, "seed": seed, "TSS": m["TSS"], "ROC-AUC": m["ROC-AUC"], "secs": secs})
            print(f"Results: Val TSS={m['TSS']:.4f} | ROC-AUC={m['ROC-AUC']:.4f} | Thr={m['Threshold']:.4f} | {secs:.0f}s")

    search_df = pd.DataFrame(rows)
    summary = search_df.groupby("config")[["TSS", "ROC-AUC"]].agg(["mean", "std"]).round(3)
    print(summary.rename(index=lambda c: c + 1))

    best_c = int(search_df.groupby("config")[select_metric].mean().idxmax())
    best_cfg = grid[best_c]
    print(f"\nBest by val {select_metric}: config #{best_c+1} | {best_cfg}")

    # Run on test set
    print(f"\nRunning {n_final_runs} final runs on test")
    print("─" * 70)
    final_rows = []
    for run in range(n_final_runs):
        seed = RAND_STATE + 100 + run
        print(f"Run {run+1} (seed {seed})...", flush=True)

        t0 = time.perf_counter()
        m = run_triplet(best_cfg, seed, X_test_t, y_test)
        secs = time.perf_counter() - t0 

        final_rows.append({"seed": seed, "TSS": m["TSS"], "ROC-AUC": m["ROC-AUC"], "Thr": m["Threshold"], "secs": secs}) 
        print(f"Mean Results | test TSS={m['TSS']:.4f} | ROC-AUC={m['ROC-AUC']:.4f} | Thr={m['Threshold']:.4f} | {secs:.0f}s") 

    final_df = pd.DataFrame(final_rows)
    print(f"\n{name} test: TSS {final_df['TSS'].mean():.3f} ± {final_df['TSS'].std():.3f} | AUC {final_df['ROC-AUC'].mean():.3f} ± {final_df['ROC-AUC'].std():.3f}")
    return {"name": name, "best_cfg": best_cfg, "search": search_df, "final": final_df}

### Setup

In [306]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

X_train_t = torch.as_tensor(X_train_norm, dtype=torch.float32)
X_val_t = torch.as_tensor(X_val_norm, dtype=torch.float32)
X_test_t = torch.as_tensor(X_test_norm, dtype=torch.float32)

device: cpu


In [307]:
BASE_CONFIG = dict(
    # loss / regularization
    margin=0.3, 
    weight_decay=0.012,
    dropout=0.0,

    # encoder
    hidden_ch=32, 
    emb_dim=32, 
    depth=6,

    # triplets
    batch_size=256,
    sep_fraction=0.5, 
    hard_frac=0.5,
    hard_pool=2000,
    refresh_every=250,
    include_hard=True,
    max_triplets=15000,

    # training
    lr=1e-3, 
    n_steps=1000, 
    eval_every=1000,

    # head specific parameters
    head_params={},
)

### Logistic Regression

In [ ]:
LOGREG_BASE = {**BASE_CONFIG, "head": "logreg", "head_params": {"C": 1e-2}}

rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []        
for i in range(15):
    cfg = {**LOGREG_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-3, -1.3), 5)
    cfg["margin"] = round(rng.uniform(0.1, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.0, 0.1, 0.2]))
    cfg["hidden_ch"] = int(rng.choice([16, 32, 64]))
    cfg["emb_dim"] = int(rng.choice([16, 32, 64]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))

    RANDOM_CFGS.append(cfg)

LOGREG_GRID = [
    {**LOGREG_BASE, "encoder": "cnn", "depth": 5, "n_steps": 0},   # untrained CNN reference
    {**LOGREG_BASE, "encoder": "gru", "depth": 2, "n_steps": 0},   # untrained GRU reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],   # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],   # random cfgs on GRU
]

RESULTS["logreg"] = run_experiment(
    name="Logistic Regression", 
    grid=LOGREG_GRID,
    n_search_seeds=1,
    n_final_runs=3,
    select_metric="TSS"
)

══════════════════════════════════════════════════════════════════════
Logistic Regression: searching 32 configs x 1 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/32 | seed 42] {'margin': 0.3, 'weight_decay': 0.012, 'dropout': 0.0, 'hidden_ch': 32, 'emb_dim': 32, 'depth': 5, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.5, 'hard_pool': 2000, 'refresh_every': 250, 'include_hard': True, 'max_triplets': 15000, 'lr': 0.001, 'n_steps': 0, 'eval_every': 1000, 'head_params': {'C': 0.01}, 'head': 'logreg', 'encoder': 'cnn'}
step     0 | loss nan | Train TSS 0.466 | Val TSS 0.560 | Train ROC-AUC 0.790 | Val ROC-AUC 0.768
Results: Val TSS=0.5595 | ROC-AUC=0.7677 | Thr=0.5602 | 11s

[config 2/32 | seed 42] {'margin': 0.3, 'weight_decay': 0.012, 'dropout': 0.0, 'hidden_ch': 32, 'emb_dim': 32, 'depth': 2, 'batch_size': 256, 'sep_fraction': 0.5, 'hard_frac': 0.5, 'hard_pool': 2000, 'refresh_every': 250, 'include_hard': True, 'max_tripl

### Centroid Distance

In [ ]:
CENTROID_BASE = {**BASE_CONFIG, "head": "centroid", "head_params": {}}

rng = np.random.default_rng(RAND_STATE + 1)

RANDOM_CFGS = []        

for i in range(15):
    cfg = {**CENTROID_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-3, -1.3), 5)
    cfg["margin"] = round(rng.uniform(0.1, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.0, 0.1, 0.2]))
    cfg["hidden_ch"] = int(rng.choice([16, 32, 64]))
    cfg["emb_dim"] = int(rng.choice([16, 32, 64]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))

    RANDOM_CFGS.append(cfg)

CENTROID_GRID = [
    {**CENTROID_BASE, "encoder": "cnn", "depth": 5, "n_steps": 0},   # untrained CNN reference
    {**CENTROID_BASE, "encoder": "gru", "depth": 2, "n_steps": 0},   # untrained GRU reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],     # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],     # random cfgs on GRU
]

RESULTS["centroid"] = run_experiment(
    name="Centroid",
    grid=CENTROID_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

══════════════════════════════════════════════════════════════════════
Centroid: searching 13 configs x 2 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/13 | seed 42] centroid {} | enc: {'n_steps': 0}
step     0 | loss nan | Train TSS 0.368 | Val TSS 0.599 | Train ROC-AUC 0.744 | Val ROC-AUC 0.821
Mean Results | Val TSS=0.5985  ROC-AUC=0.8215  Thr=0.7572

[config 1/13 | seed 43] centroid {} | enc: {'n_steps': 0}
step     0 | loss nan | Train TSS 0.349 | Val TSS 0.461 | Train ROC-AUC 0.736 | Val ROC-AUC 0.689
Mean Results | Val TSS=0.4609  ROC-AUC=0.6890  Thr=0.6088

[config 2/13 | seed 42] centroid {} | enc: base
step     0 | loss nan | Train TSS 0.368 | Val TSS 0.599 | Train ROC-AUC 0.744 | Val ROC-AUC 0.821
step  1000 | loss 0.0962 | Train TSS 0.702 | Val TSS 0.706 | Train ROC-AUC 0.909 | Val ROC-AUC 0.861
    d_ap 0.213 | d_an 0.308 | active 0.48 | spread 0.270
Mean Results | Val TSS=0.7059  ROC-AUC=0.8611  Thr=0.7021

[config 2

### kNN

In [ ]:
KNN_BASE = {**BASE_CONFIG, "head": "knn"}

rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []        
for i in range(15):
    cfg = {**KNN_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-3, -1.3), 5)
    cfg["margin"] = round(rng.uniform(0.1, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.0, 0.1, 0.2]))
    cfg["hidden_ch"] = int(rng.choice([16, 32, 64]))
    cfg["emb_dim"] = int(rng.choice([16, 32, 64]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))
    cfg["head_params"] = {"k": int(rng.choice([5, 10, 20, 40, 80]))}

    RANDOM_CFGS.append(cfg)

KNN_GRID = [
    {**KNN_BASE, "encoder": "cnn", "depth": 5, "n_steps": 0},        # untrained CNN reference
    {**KNN_BASE, "encoder": "gru", "depth": 2, "n_steps": 0},        # untrained GRU reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],     # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],     # random cfgs on GRU
]

RESULTS["knn"] = run_experiment(
    name="kNN",
    grid=KNN_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

══════════════════════════════════════════════════════════════════════
kNN: searching 14 configs x 2 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/14 | seed 42] knn {'k': 5} | enc: {'n_steps': 0}
step     0 | loss nan | Train TSS 0.077 | Val TSS 0.205 | Train ROC-AUC 0.524 | Val ROC-AUC 0.524
Mean Results | Val TSS=0.2045 | ROC-AUC=0.5238 | Thr=0.3397

[config 1/14 | seed 43] knn {'k': 5} | enc: {'n_steps': 0}
step     0 | loss nan | Train TSS 0.138 | Val TSS 0.254 | Train ROC-AUC 0.562 | Val ROC-AUC 0.589
Mean Results | Val TSS=0.2541 | ROC-AUC=0.5886 | Thr=0.5213

[config 2/14 | seed 42] knn {'k': 1} | enc: base
step     0 | loss nan | Train TSS 0.061 | Val TSS 0.262 | Train ROC-AUC 0.504 | Val ROC-AUC 0.562
step  1000 | loss 0.1483 | Train TSS 0.994 | Val TSS 0.378 | Train ROC-AUC 0.997 | Val ROC-AUC 0.691
Mean Results | Val TSS=0.3778 | ROC-AUC=0.6909 | Thr=0.5934

[config 2/14 | seed 43] knn {'k': 1} | enc: base
step     0 | 

KeyboardInterrupt: 

### SVM

In [ ]:
SVM_BASE = {**BASE_CONFIG, "head": "svm", "head_params": {"kernel": "rbf", "C": 1.0, "gamma": 0.01}}

rng = np.random.default_rng(RAND_STATE + 1)
RANDOM_CFGS = []        
for i in range(15):
    cfg = {**SVM_BASE}
    cfg["lr"] = round(10 ** rng.uniform(-4, -2.5), 6)
    cfg["weight_decay"] = round(10 ** rng.uniform(-3, -1.3), 5)
    cfg["margin"] = round(rng.uniform(0.1, 0.6), 3)
    cfg["dropout"] = float(rng.choice([0.0, 0.1, 0.2]))
    cfg["hidden_ch"] = int(rng.choice([16, 32, 64]))
    cfg["emb_dim"] = int(rng.choice([16, 32, 64]))
    cfg["hard_frac"] = round(rng.uniform(0.0, 0.8), 2)
    cfg["hard_pool"] = int(10 ** rng.uniform(2.3, 3.5))
    cfg["include_hard"] = bool(rng.choice([False, True]))
    cfg["head_params"] = {
        "kernel": "rbf",
        "C": 10 ** rng.uniform(-2, 1),
        "gamma": 10 ** rng.uniform(-3, -1),
        "standardize": bool(rng.choice([True, False])),
    }

    RANDOM_CFGS.append(cfg)

SVM_GRID = [
    {**SVM_BASE, "encoder": "cnn", "depth": 5, "n_steps": 0},        # untrained CNN reference
    {**SVM_BASE, "encoder": "gru", "depth": 2, "n_steps": 0},        # untrained GRU reference
    *[{**c, "encoder": "cnn", "depth": 5} for c in RANDOM_CFGS],     # random cfgs on CNN
    *[{**c, "encoder": "gru", "depth": 2} for c in RANDOM_CFGS],     # random cfgs on GRU
]

RESULTS["svm"] = run_experiment(
    name="SVM",
    grid=SVM_GRID,
    n_search_seeds=2,
    n_final_runs=3,
    select_metric="TSS",
)

══════════════════════════════════════════════════════════════════════
SVM: searching 19 configs x 2 seeds on validation
══════════════════════════════════════════════════════════════════════

[config 1/19 | seed 42] svm {'kernel': 'rbf', 'C': 1.0} | enc: {'n_steps': 0}
step     0 | loss nan | Train TSS 0.975 | Val TSS 0.573 | Train ROC-AUC 0.995 | Val ROC-AUC 0.834
Mean Results | Val TSS=0.5726  ROC-AUC=0.8342  Thr=0.6999

[config 1/19 | seed 43] svm {'kernel': 'rbf', 'C': 1.0} | enc: {'n_steps': 0}
step     0 | loss nan | Train TSS 0.977 | Val TSS 0.411 | Train ROC-AUC 0.994 | Val ROC-AUC 0.740
Mean Results | Val TSS=0.4110  ROC-AUC=0.7399  Thr=0.4910

[config 2/19 | seed 42] svm {'kernel': 'linear', 'C': 0.01} | enc: base
step     0 | loss nan | Train TSS 0.511 | Val TSS 0.651 | Train ROC-AUC 0.815 | Val ROC-AUC 0.834
step  1000 | loss 0.0962 | Train TSS 0.740 | Val TSS 0.622 | Train ROC-AUC 0.921 | Val ROC-AUC 0.858
    d_ap 0.213 | d_an 0.308 | active 0.48 | spread 0.270
Mean Resu

# InfoNCE

In [ ]:
X_train_t = torch.as_tensor(X_train_norm, dtype=torch.float32)
X_val_t = torch.as_tensor(X_val_norm, dtype=torch.float32)
X_test_t = torch.as_tensor(X_test_norm, dtype=torch.float32)

y_train_t = torch.as_tensor(y_train, dtype=torch.long)
y_val_t = torch.as_tensor(y_val, dtype=torch.long)
y_test_t = torch.as_tensor(y_test, dtype=torch.long)

# ---- Model (outputs L2-normalized embeddings) ----
class Encoder(nn.Module):
    def __init__(self, n_features, embed_dim=32, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, embed_dim)

    def forward(self, x):
        _, h = self.gru(x)
        return F.normalize(self.fc(h[-1]), dim=1)

# ---- Augmentation: jitter + per-channel scaling ----
def augment(x):
    noise = 0.1 * torch.randn_like(x)
    scale = 1 + 0.1 * torch.randn(x.shape[0], 1, x.shape[2], device=x.device)
    return x * scale + noise

# ---- Supervised contrastive loss ----
def supcon_loss(z, y, temp=0.1):
    sim = z @ z.T / temp                                   # (2B, 2B)
    eye = torch.eye(len(y), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, -1e9)                       # ignore self-similarity
    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos = (y[:, None] == y[None, :]) & ~eye                # same label, not self
    return -(log_prob.masked_fill(~pos, 0).sum(1) / pos.sum(1)).mean()

# ---- Evaluation helpers ----
def embed(model, X, batch_size=512):
    model.eval()
    with torch.no_grad():
        out = [model(X[i:i + batch_size]).cpu() for i in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()

def tss(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tp / (tp + fn) + tn / (tn + fp) - 1

def predict_proba(model, X_ref, y_ref, X_eval, k=50):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(embed(model, X_ref), y_ref.numpy())            # reference = real (unbalanced) training data
    return knn.predict_proba(embed(model, X_eval))[:, 1]

thresholds = np.linspace(0.01, 0.99, 99)

def best_threshold(y_true, p):
    scores = [tss(y_true, (p >= t).astype(int)) for t in thresholds]
    i = int(np.argmax(scores))
    return thresholds[i], scores[i]

# ---- Balanced batches (roughly 50/50) ----
class_counts = torch.bincount(y_train_t)
weights = (1.0 / class_counts.float())[y_train_t]
sampler = WeightedRandomSampler(weights, num_samples=len(y_train_t), replacement=True)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, sampler=sampler)

# ---- Training ----
model = Encoder(n_features=X_train_t.shape[-1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_val_tss, best_state, best_thr = -1.0, None, 0.5

for epoch in range(20):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        z = model(torch.cat([augment(xb), augment(xb)]))   # two views -> (2B, D)
        loss = supcon_loss(z, torch.cat([yb, yb]))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    p_val = predict_proba(model, X_train_t, y_train_t, X_val_t)
    val_auc = roc_auc_score(y_val_t.numpy(), p_val)
    thr, val_tss = best_threshold(y_val_t.numpy(), p_val)

    if val_tss > best_val_tss:
        best_val_tss, best_state, best_thr = val_tss, copy.deepcopy(model.state_dict()), thr

    print(f"epoch {epoch}: loss {train_loss:.4f} | val AUC {val_auc:.3f} | "
          f"val TSS {val_tss:.4f} @ thr {thr:.2f}")

# ---- Final test with best checkpoint and validation-chosen threshold ----
model.load_state_dict(best_state)
p_test = predict_proba(model, X_train_t, y_train_t, X_test_t)
test_tss = tss(y_test_t.numpy(), (p_test >= best_thr).astype(int))
test_auc = roc_auc_score(y_test_t.numpy(), p_test)
print(f"best val TSS {best_val_tss:.4f} (thr {best_thr:.2f}) | test TSS {test_tss:.4f} | test AUC {test_auc:.3f}")

epoch 0: loss 6.4866 | val AUC 0.724 | val TSS 0.3890 @ thr 0.01
epoch 1: loss 6.1562 | val AUC 0.793 | val TSS 0.5371 @ thr 0.01
epoch 2: loss 6.1342 | val AUC 0.805 | val TSS 0.5121 @ thr 0.01
epoch 3: loss 6.1212 | val AUC 0.780 | val TSS 0.5285 @ thr 0.03
epoch 4: loss 6.1131 | val AUC 0.799 | val TSS 0.5902 @ thr 0.03
epoch 5: loss 6.0985 | val AUC 0.718 | val TSS 0.4086 @ thr 0.03
epoch 6: loss 6.0830 | val AUC 0.777 | val TSS 0.5038 @ thr 0.01
epoch 7: loss 6.0617 | val AUC 0.776 | val TSS 0.4992 @ thr 0.01
epoch 8: loss 6.0492 | val AUC 0.745 | val TSS 0.4586 @ thr 0.01
epoch 9: loss 6.0304 | val AUC 0.753 | val TSS 0.4830 @ thr 0.01
epoch 10: loss 6.0166 | val AUC 0.729 | val TSS 0.4332 @ thr 0.01
epoch 11: loss 5.9874 | val AUC 0.702 | val TSS 0.3602 @ thr 0.01
epoch 12: loss 5.9336 | val AUC 0.672 | val TSS 0.3422 @ thr 0.01
epoch 13: loss 5.9035 | val AUC 0.571 | val TSS 0.1472 @ thr 0.01
epoch 14: loss 5.8757 | val AUC 0.614 | val TSS 0.2259 @ thr 0.01
epoch 15: loss 5.820